# 03a - Light metrics (Quote-CER, Concept-Rec, BLEU-4, METEOR)

**Mục tiêu:** chấm nhanh & ổn định từ file predictions CSV (detail-only), *không gọi API*.

**Inputs**
- `PRED_CSV`: file dự đoán từ notebook 02  
  (ví dụ: `baseline_predictions_test_detail.csv`, `A_predictions_test_detail.csv`, ...)

**Outputs** *(trong `OUT_DIR`)*  
- `{RUN_ID}_metrics_light.json` : `{quote_cer, concept_rec, bleu4, meteor, n, model_name}`
- `{RUN_ID}_main_results_row_light.csv` : 1 dòng để tiện copy vào bảng kết quả

⚠️ Lưu ý khi chạy Save & Run All (Commit) trên Kaggle
- Notebook này có bước **cài thư viện bằng pip** và **tải NLTK data** (`nltk.download(...)`). Khi chạy **Commit** (headless), nếu Notebook **tắt Internet** hoặc môi trường bị giới hạn tải xuống, notebook có thể **lỗi ở bước cài deps / download WordNet**.
- Khuyến nghị: chạy Interactive với Internet ON để tạo `{RUN_ID}_metrics_light.json` / `{RUN_ID}_main_results_row_light.csv`.

**Notebook kế tiếp:** chạy 03B (heavy metrics) rồi notebook 04 để merge light + heavy.


## 0) Config

In [ ]:
# === USER CONFIG ===
RUN_ID = "B"  # baseline | A | B
PRED_CSV = f"/kaggle/input/vn-textbook-qwen2vl-02-predictions/{RUN_ID}_predictions_test_detail.csv"
OUT_DIR  = "/kaggle/working"

# (tuỳ chọn) name hiển thị trong bảng kết quả
if RUN_ID == "baseline":
    MODEL_NAME = "Qwen2-VL-2B (baseline)" 
else:
    MODEL_NAME = f"Qwen2-VL-2B (FT-{RUN_ID})" 

# Concept-Rec: POS tags lấy danh từ + số từ (theo report)
CONCEPT_POS_KEEP = {"N", "Np", "Nc", "Nu", "M"}  # underthesea tagset

# BLEU tokenization (sacrebleu)
SACREBLEU_TOKENIZE = "intl"

## 1) Install deps (metrics only)
Cell này **có thể dừng (SystemExit)** khi chạy interactive để bạn restart kernel sau pip install.
Khi **Save & Commit (batch run)** thì notebook sẽ tiếp tục chạy bình thường.

In [ ]:
import os, pathlib

MARK = pathlib.Path("/kaggle/working/.deps_installed_light_metrics")
run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower()
is_batch = run_type in ("batch", "commit", "run")
is_interactive = not is_batch

if not MARK.exists():
    # Pin numpy/scipy để tránh mismatch binary khi underthesea->nltk->scipy import
    !pip -q install --no-cache-dir -U --force-reinstall "numpy>=2.0,<3" "scipy>=1.11,<2"
    !pip -q install --no-cache-dir -U underthesea nltk sacrebleu pandas tqdm

    MARK.write_text("ok")

    # QUAN TRỌNG:
    # - Interactive: sau pip install, restart kernel là cách chắc nhất để tránh lỗi ABI numpy/scipy.
    # - Batch/Commit: kernel luôn fresh, có thể tiếp tục luôn.
    if is_interactive:
        raise SystemExit("Deps installed. PLEASE Restart Session/Kernel rồi chạy lại từ đầu để tránh lỗi ABI numpy/scipy.")
    else:
        print("Deps installed in batch run. Continuing (fresh env, no restart needed).")
else:
    print("Deps already installed (marker found).")

# Chỉ import NLTK sau khi deps đã ổn định (marker tồn tại hoặc đang chạy batch fresh)
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
print("NLTK data ready.")

## 2) Load predictions

In [ ]:
import pandas as pd
import numpy as np

pred_df = pd.read_csv(PRED_CSV)
print("Loaded:", pred_df.shape)
print("Columns:", pred_df.columns.tolist())

# auto-detect columns
def pick_col(cands):
    for c in cands:
        if c in pred_df.columns:
            return c
    return None

COL_PRED = pick_col(["pred_detail", "pred", "prediction", "pred_text"])
COL_REF  = pick_col(["gt_detail", "reference", "ref", "gt_text"])
COL_ID   = pick_col(["id", "sample_id"])

assert COL_PRED is not None, "Missing prediction column (expected one of: pred_detail/pred/prediction/pred_text)"
assert COL_REF  is not None, "Missing reference column (expected one of: gt_detail/reference/ref/gt_text)"

preds = pred_df[COL_PRED].fillna("").astype(str).tolist()
refs  = pred_df[COL_REF ].fillna("").astype(str).tolist()

print("Using:", {"pred": COL_PRED, "ref": COL_REF, "id": COL_ID})
print("n =", len(preds))

## 3) Metrics: Quote-CER (↓)

In [ ]:
import re

def _norm_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

# simple Levenshtein edit distance (two-row DP)
def _lev(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)
    # ensure b is shorter for memory
    if len(b) > len(a):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (0 if ca == cb else 1)
            cur.append(min(ins, dele, sub))
        prev = cur
    return prev[-1]

# extract quoted spans: "..." or “...” or '...'
QUOTE_PATTERNS = [
    r'"([^"]+)"',
    r'“([^”]+)”',
    r"‘([^’]+)’",
    r"'([^']+)'",
]
QUOTE_RE = re.compile("|".join(QUOTE_PATTERNS))

def extract_quotes(text: str):
    text = _norm_ws(text)
    out = []
    for m in QUOTE_RE.finditer(text):
        # m.groups() contains many Nones, pick first non-None
        g = next((x for x in m.groups() if x is not None), None)
        if g:
            g = _norm_ws(g)
            if g:
                out.append(g)
    return out

def quote_cer_one(ref_text: str, pred_text: str):
    ref_q = extract_quotes(ref_text)
    pred_q = extract_quotes(pred_text)
    # if no quotes in ref => define CER=0 (no OCR quote requirement)
    if len(ref_q) == 0:
        return 0.0, 0, 0

    # greedy match each ref quote to best pred quote (min edit distance)
    used = set()
    total_edits = 0
    total_chars = 0
    for rq in ref_q:
        total_chars += len(rq)
        best = None
        best_j = None
        for j, pq in enumerate(pred_q):
            if j in used:
                continue
            d = _lev(rq, pq)
            if (best is None) or (d < best):
                best = d
                best_j = j
        if best is None:
            # missing quote => all chars are errors (delete)
            total_edits += len(rq)
        else:
            used.add(best_j)
            total_edits += best
    cer = total_edits / max(1, total_chars)
    return cer, total_edits, total_chars

cers = []
total_ed, total_ch = 0, 0
for r, p in zip(refs, preds):
    cer, ed, ch = quote_cer_one(r, p)
    cers.append(cer)
    total_ed += ed
    total_ch += ch

quote_cer = total_ed / max(1, total_ch)  # corpus-style
print("Quote-CER:", quote_cer, f"(total_edits={total_ed}, total_chars={total_ch})")

## 4) Metrics: Concept-Rec (↑)

In [ ]:
from underthesea import pos_tag
from tqdm.auto import tqdm

def extract_concepts(text: str):
    # lower + normalize whitespace
    text = _norm_ws(text).lower()
    if not text:
        return set()
    try:
        tags = pos_tag(text)  # list[(token, tag)]
    except Exception:
        return set()
    concepts = set()
    for tok, tag in tags:
        if tag in CONCEPT_POS_KEEP:
            tok = _norm_ws(tok)
            if tok:
                concepts.add(tok)
    return concepts

def concept_recall_one(ref_text: str, pred_text: str):
    ref_c = extract_concepts(ref_text)
    if len(ref_c) == 0:
        return 1.0
    pred_c = extract_concepts(pred_text)
    hit = len(ref_c & pred_c)
    return hit / len(ref_c)

recs = [concept_recall_one(r, p) for r, p in tqdm(list(zip(refs, preds)), desc="Concept-Rec")]
concept_rec = float(np.mean(recs))
print("Concept-Rec:", concept_rec)

## 5) Metrics: BLEU-4 (↑) + METEOR (↑)

In [ ]:
import sacrebleu
import nltk
from nltk.translate.meteor_score import meteor_score
from tqdm.auto import tqdm

# BLEU-4 (corpus)
bleu = sacrebleu.corpus_bleu(preds, [refs], tokenize=SACREBLEU_TOKENIZE)
bleu4 = float(bleu.score)  # 0..100
print("BLEU-4:", bleu4)

# METEOR (mean sentence-level) — requires wordnet + omw-1.4
def _meteor_one(r: str, p: str) -> float:
    r_tok = _norm_ws(r).split()
    p_tok = _norm_ws(p).split()
    if len(r_tok) == 0 and len(p_tok) == 0:
        return 1.0
    if len(r_tok) == 0:
        return 0.0
    try:
        return float(meteor_score([r_tok], p_tok))
    except LookupError:
        # try download then retry once
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        return float(meteor_score([r_tok], p_tok))

meteor_scores = [_meteor_one(r, p) for r, p in tqdm(list(zip(refs, preds)), desc="METEOR")]
meteor = float(np.mean(meteor_scores))
print("METEOR:", meteor)

## 6) Save outputs (light metrics)

In [ ]:
import json, os
from pathlib import Path

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

metrics_light = {
    "model_name": MODEL_NAME,
    "n": len(preds),
    "quote_cer": float(quote_cer),
    "concept_rec": float(concept_rec),
    "bleu4": float(bleu4),
    "meteor": float(meteor),
}

(out_dir / f"{RUN_ID}_metrics_light.json").write_text(json.dumps(metrics_light, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", out_dir / f"{RUN_ID}_metrics_light.json")

# Row template for LaTeX table (leave heavy fields blank; notebook B will fill)
import pandas as pd
row = {
    "Model": MODEL_NAME,
    "Quote-CER": float(quote_cer),
    "Concept-Rec": float(concept_rec),
    "LLM-Score": "",
    "BERTScore": "",
    "BLEU-4": float(bleu4),
    "METEOR": float(meteor),
}
pd.DataFrame([row]).to_csv(out_dir / f"{RUN_ID}_main_results_row_light.csv", index=False)
print("Wrote:", out_dir / f"{RUN_ID}_main_results_row_light.csv")
